## Neural Dimensionality Estimation

**Created:** Thursday, January 16, 2026, 14:58:05  
**Author:** @nehabinish  

This notebook estimates the dimensionality of neural ECoG data for individual subjects using Factor Analysis (FA).  

In [13]:
#%% == Autoreload ==

# Run cell to enable autoreload of modules
%load_ext autoreload
%autoreload 2

In [18]:
# %% == Get working dir to set path ==

import os
import sys

try:
    from pathlib import Path # Try using pathlib 
    cwd = Path.cwd()         # Current working directory
    parent_dir = cwd.parent  # Parent directory

    # Add parent directory to system path
    sys.path.append(str(parent_dir))
    sys.path.append(str(parent_dir / 'src/'))

except Exception as e:
    # Fallback using os module if pathlib fails
    try:
        cwd = os.getcwd()
        parent_dir = os.path.dirname(cwd)

        sys.path.append(parent_dir)
        sys.path.append(os.path.join(parent_dir, 'src'))
    except Exception as ex:
        print("Could not determine workspace paths automatically.")
        print("Please ensure you are running the notebook in a proper directory.")
        print("If error persists, manually set cwd and parent_dir.")
        print("Error details:", ex)

# Notes for users:
# 1. Make sure your notebook is launched from the root project folder
#    so that the parent directory contains your modules.
# 2. If cell fails, manually set 'cwd' and 'parent_dir' at the top.
# 3. This setup is crucial for importing custom modules located in the parent directory.

In [22]:
# %% == Import modules ==

# Python standard libraries
import pickle
import numpy as np
import matplotlib.pyplot as plt

# Custom modules
import init as In
from dimreduction_utils import FactorAnalysis

# Set random seeds for reproducibility
rng = np.random.default_rng(42)

In [25]:
# %% == Load example data ==

data_path = os.path.join(parent_dir, 'data', 'demo_data.pkl')
with open(data_path, 'rb') as f:
    data = pickle.load(f)

In [26]:
# %% == Function to check data dimensionality ==

def find_data_dimensionality(data):
    """
    Estimates the required number of latent dimensions for the given dataset 
    using cross-validated Factor Analysis (FA).
    
    Args:
        - data (np.ndarray): A 2D array of shape [samples x features].
    Returns:
        - qOpt (int): The estimated number of latent dimensions for the data.
        - expvar (np.ndarray): Explained variance for each latent dimension.
    """

    # Ensure the input is a 2D array
    data = np.array(data)
    if data.ndim != 2:
        raise ValueError("Input data must be a 2D array with shape [samples x features].")
    nsamples, nfeatures = data.shape

    # Ensure there are more samples than features
    if nsamples < nfeatures:
        raise ValueError("Number of samples must be greater than or equal to the " \
        "number of features (nsamples >= nfeatures).")

    # Number of latent dimensions to test 
    q = np.arange(0, nfeatures+1, 1) 

    # Initialize Factor Analysis class and perform cross-validated FA
    FA = FactorAnalysis(X=data, q=q)
    cvLoss, cvLogLike, expvar  = FA.CrossValFa()

    # Select the optimal dimensionality based on cross-validation loss
    qOpt = FA.FactorAnalysisModelSelect(cvLoss)

    return qOpt, expvar


In [31]:
# %% == Estimate data dimensionality ==

nTrials = len(data['trial_type'])   # Number of trials in the dataset
R1_trials = data['R1']              # Trial data for brain region R1
R2_trials = data['R2']              # Trial data for brain region R2
nRegions = 2                        # Number of brain regions (R1 and R2)

Results = {}

# Loop over single trials
for r_idx in range(nRegions):
    print(f"Processing Region {r_idx + 1}...")
     
    if r_idx == 0:
        region_data = R1_trials
        region_name = 'R1'
    else:
        region_data = R2_trials
        region_name = 'R2'

    single_trial_dims, single_trial_expvar = [], []
    for trial_no in range(nTrials):
        print(f"  Analyzing Trial {trial_no + 1} / {nTrials}...")
        region_single_trial = region_data[trial_no, :, :]

        dim, expvar = find_data_dimensionality(region_single_trial.T)
        single_trial_dims.append(dim)
        single_trial_expvar.append(expvar)
        
    Results[region_name] = {
        'dims': single_trial_dims,
        'expvar': single_trial_expvar
    }

Processing Region 1...
  Analyzing Trial 1 / 20...


  Analyzing Trial 2 / 20...
  Analyzing Trial 3 / 20...
  Analyzing Trial 4 / 20...
  Analyzing Trial 5 / 20...
  Analyzing Trial 6 / 20...
  Analyzing Trial 7 / 20...
  Analyzing Trial 8 / 20...
  Analyzing Trial 9 / 20...
  Analyzing Trial 10 / 20...
  Analyzing Trial 11 / 20...
  Analyzing Trial 12 / 20...
  Analyzing Trial 13 / 20...
  Analyzing Trial 14 / 20...
  Analyzing Trial 15 / 20...
  Analyzing Trial 16 / 20...
  Analyzing Trial 17 / 20...
  Analyzing Trial 18 / 20...
  Analyzing Trial 19 / 20...
  Analyzing Trial 20 / 20...
Processing Region 2...
  Analyzing Trial 1 / 20...
  Analyzing Trial 2 / 20...
  Analyzing Trial 3 / 20...
  Analyzing Trial 4 / 20...
  Analyzing Trial 5 / 20...
  Analyzing Trial 6 / 20...
  Analyzing Trial 7 / 20...
  Analyzing Trial 8 / 20...
  Analyzing Trial 9 / 20...
  Analyzing Trial 10 / 20...
  Analyzing Trial 11 / 20...
  Analyzing Trial 12 / 20...
  Analyzing Trial 13 / 20...
  Analyzing Trial 14 / 20...
  Analyzing Trial 15 / 20...
  Analyz

In [32]:
print(Results['R1'])

{'dims': [0, np.int64(2), 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, np.int64(1), 0, 0, 0, 0, 0, 0], 'expvar': [nan, array([ 5.60771365e-01,  4.39228635e-01,  7.22270682e-17,  6.12242424e-17,
        1.00960042e-17,  7.52768419e-18,  6.55589192e-18,  4.44001865e-18,
        3.37009737e-18,  2.23942536e-18,  1.39681012e-18,  7.65560414e-19,
       -2.33845320e-19, -4.97822808e-19, -1.68620828e-18, -2.44651139e-18,
       -3.54505661e-18, -7.21719263e-18, -8.84280571e-18, -1.65102013e-17,
       -2.07889867e-17, -3.25926994e-17]), nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, array([ 1.00000000e+00,  2.33234516e-16,  4.26321096e-18,  6.25150676e-19,
        3.88967607e-19,  1.44698219e-19,  6.30055080e-20,  3.04769867e-20,
        2.54755942e-20,  9.11374569e-21,  6.38876216e-21,  6.03615801e-22,
       -8.48233191e-21, -1.45865827e-20, -2.86886501e-20, -5.85411926e-20,
       -9.47068966e-20, -1.47823481e-19, -2.60875882e-19, -3.82788634e-19,
       -5.80204337e-19, -1.36851377e-17]), na